## cleaning pipeline

Data inconsistencies across vehicle IDs, odometer
units, timestamps, and rate plans affect utilization, pricing, and customer experience.

In [2]:
import pandas as pd 
import numpy as np

In [3]:
df=pd.read_csv("car_rental.csv")
    

## 1. Vehicle_ID trimming and canonical case.


In [4]:
# 1.
df['Vehicle_ID'].unique()
# checking for null values in the 'Vehicle_ID' column
df['Vehicle_ID'].isnull().sum() # no null values

# Converting to uppercase
df['Vehicle_ID'] = df['Vehicle_ID'].str.upper()

# Removing leading and trailing spaces
df['Vehicle_ID'] = df['Vehicle_ID'].str.strip()  

# Replacing spaces with underscores
df['Vehicle_ID'] = df['Vehicle_ID'].str.replace( ' ', '-') 
 
# Checking the unique values after cleaning
print(len(df['Vehicle_ID'].unique()))

# checking the unique values after cleaning
df['Vehicle_ID'].unique()

30


<StringArray>
['CAR-18', 'CAR-12', 'CAR-28', 'CAR-24', 'CAR-07', 'CAR-16', 'CAR-22',
 'CAR-20', 'CAR-02', 'CAR-25', 'CAR-03', 'CAR-26', 'CAR-06', 'CAR-05',
 'CAR-09', 'CAR-27', 'CAR-08', 'CAR-11', 'CAR-15', 'CAR-23', 'CAR-19',
 'CAR-13', 'CAR-01', 'CAR-10', 'CAR-04', 'CAR-30', 'CAR-17', 'CAR-29',
 'CAR-14', 'CAR-21']
Length: 30, dtype: str

## 2. Timestamp normalization; invalid minutes fix; duration must be positive.


In [5]:
 #2. Convert to datetime (auto handles overflow like 75 sec)
def fix_timestamp(col):
    return pd.to_datetime(col, errors='coerce')

# Apply to columns
df["Booking_TS"] = fix_timestamp(df["Booking_TS"])
df["Pickup_TS"] = fix_timestamp(df["Pickup_TS"])
df["Return_TS"] = fix_timestamp(df["Return_TS"])

# -------------------------------
# Fix invalid timestamps manually (fallback)
# -------------------------------

def normalize_time(ts):
    if pd.isna(ts):
        return ts
    
    # extract components
    sec = ts.second
    minute = ts.minute
    hour = ts.hour

    # fix seconds overflow
    minute += sec // 60
    sec = sec % 60

    # fix minutes overflow
    hour += minute // 60
    minute = minute % 60

    return ts.replace(hour=hour % 24, minute=minute, second=sec)

# Apply normalization
df["Pickup_TS"] = df["Pickup_TS"].apply(normalize_time)
df["Return_TS"] = df["Return_TS"].apply(normalize_time)
df["Booking_TS"] = df["Booking_TS"].apply(normalize_time)




# -------------------------------
# STEP 3: Handle NaT in Pickup_TS
# -------------------------------

# Fill missing Pickup_TS using Booking_TS
df["Pickup_TS"] = df["Pickup_TS"].fillna(df["Booking_TS"])

# Drop remaining NaT (if any)
df = df.dropna(subset=["Pickup_TS"])

print(df[["Pickup_TS", "Return_TS", "Booking_TS"]].head())

            Pickup_TS           Return_TS          Booking_TS
0 2026-03-23 03:01:00 2026-03-24 18:01:00 2026-03-22 13:01:00
1 2026-02-25 11:17:00 2026-03-01 14:17:00 2026-02-25 11:17:00
2 2026-02-26 18:07:00 2026-03-04 00:07:00 2026-02-26 18:07:00
3 2026-02-05 14:40:00 2026-02-09 12:40:00 2026-02-04 11:40:00
4 2026-02-18 00:59:00 2026-02-21 23:59:00 2026-02-18 00:59:00


C:\Users\Administrator\AppData\Local\Temp\ipykernel_29760\2713467237.py:3: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(col, errors='coerce')


## 3. Odometer numeric extraction and unit unification (km).


In [6]:
#3
# Odo_Start

# checking the unique values  column
df['Odo_Start'].unique()

# checking for null values
df['Odo_Start'].isnull().sum() # no null values

# removing commas
df['Odo_Start'] = df['Odo_Start'].str.replace(',', '')

# removing 'km'
df['Odo_Start'] = df['Odo_Start'].str.replace('km','')

# removing leading and trailing spaces
df['Odo_Start'] = df['Odo_Start'].str.strip()

# converting to int
df['Odo_Start'] = df['Odo_Start'].astype(int)

# checking the unique values after cleaning
df['Odo_Start'].unique()

# Odo_End

# checking the unique values in the 'Odo_End' column
df['Odo_End'].unique()

# checking for null values in the 'Odo_End' column
df['Odo_End'].isnull().sum() # no null values

# removing commas
df['Odo_End'] = df['Odo_End'].str.replace(',', '')

# removing 'km'
df['Odo_End'] = df['Odo_End'].str.replace('km','')

# removing leading and trailing spaces
df['Odo_End'] = df['Odo_End'].str.strip()

# converting to int
df['Odo_End'] = df['Odo_End'].astype(int)

# checking the unique values after cleaning
df['Odo_End'].unique()

array([ 30857,  41829, 101678,  67079,  42647,  41551, 117535,  66777,
        64884, 118493,  68669,  36212,   8998,  87520,  50754,  86601,
        55765,  30433,  62312,  33521,  12400,  54991,  71934,  44200,
        45906,  26926,  40555,  39285,  81991,  70989,  48058,  95289,
        50736,  19558,  55774, 107657, 120745,  95953,  29976,  78842,
        62722, 113233, 109452,  87502,   7096,  37777,  42710,  16670,
       109879,  78968,  67164,  68044,  71075,  54379,  57946,  38060,
        22991, 112315,  13150,  57616, 117183,  81431,  83139,  22049,
        34744, 118956,  71846,  67029,  29240,  15214,   8570, 101867,
        87312, 115484,  42522,  13176,  13309, 106468,  57652,  30690,
        77589, 111988,  95837,  57253, 117984,  72368,  46413,  39544,
        30097,  32068, 101004,  76721,  56928,  46371, 109368,   6762,
        60258, 117067, 103855,  41480,  44142,  46231,  55456,  18483,
        17280,  29819,  51315, 107917,  74461,  82220,  25349,  38421,
      

## 4. Fuel level normalization (50%→0.5).


In [7]:
# 'Fuel_Level' column

# checking the unique values 
df['Fuel_Level'].unique()

# checking for null values 
df['Fuel_Level'].isnull().sum() # no null values

# removing '%'
df['Fuel_Level'] = df['Fuel_Level'].str.replace('%', '')

# removing leading and trailing spaces
df['Fuel_Level'] = df['Fuel_Level'].str.strip()

# converting to float
df['Fuel_Level'] = df['Fuel_Level'].astype(float)

# converting values greater than 1 to a fraction (50 → 0.50)
df['Fuel_Level'] = df['Fuel_Level'].apply(lambda x: x/100 if x > 1 else x)

# formatting values to 2 decimal places
df['Fuel_Level'] = df['Fuel_Level'].map(lambda x: f"{x:.2f}")

# converting to float
df['Fuel_Level'] = df['Fuel_Level'].astype(float)

# renaming 'Fuel_Level' to 'Fuel_Fraction'
df.rename(columns={'Fuel_Level': 'Fuel_Fraction'}, inplace=True)

# checking the unique values after cleaning
df['Fuel_Fraction'].unique()

# checking for null values 
df['Fuel_Fraction'].isnull().sum() # no null values

np.int64(0)

## 5. Rate parsing to numeric daily rate; currency normalization.


In [8]:
#5. Rate parsing to numeric daily rate; currency normalization.
# checking the unique values in the 'Rate' column
df['Rate'].unique()

# removing '₹' symbol
df['Rate'] = df['Rate'].str.replace('₹', '')

# removing '/day'
df['Rate'] = df['Rate'].str.replace('/day','')

# removing leading and trailing spaces
df['Rate'] = df['Rate'].str.strip()

# removing commas
df['Rate'] = df['Rate'].str.replace(',', '')

# converting to int
df['Rate'] = df['Rate'].astype(int)

# renameing 'Rate' to 'Daily_Rate'
df.rename(columns={'Rate':'Daily_Rate'},inplace=True)

# checking the unique values after cleaning
df['Daily_Rate'].unique()

array([1500,  800, 2500, 1200, 2000, 1000, 3000])

## 6. City normalization to canonical names.


In [9]:
# 6. City normalization to canonical names.
df['City'].unique()
'''['Kolkata', 'delhi', 'Chennai', 'kolkata', 'Mumbai', 'CHENNAI',
       'Hyderabad', 'BENGALURU', 'mumbai', 'Delhi', 'blr', 'Bengaluru',
       'HYD', 'bangalore'],'''

dict_change_city = {
    'delhi': 'Delhi',
    'kolkata': 'Kolkata',
    'CHENNAI': 'Chennai',
    'mumbai': 'Mumbai',
    'BENGALURU': 'bangalore',
    'blr': 'bangalore',
    'Bengaluru': 'bangalore',
    'HYD': 'Hyderabad'
}

# standardizing city names
df['City'] = df['City'].replace(dict_change_city)

## 7. Duplicate reservation dedup (same ID).


In [10]:
#7. Duplicate reservation dedup (same ID).
# checking duplicate reservation IDs
df['Reservation_ID'].duplicated().sum() # 280 duplicate reservation IDs found.

# sorting by Pickup_TS to keep latest correct record
df = df.sort_values(by='Pickup_TS')

# removing duplicate records (keeping first occurrence)
df = df.drop_duplicates(subset='Reservation_ID', keep='first')

# checking duplicates after removal
df['Reservation_ID'].duplicated().sum() #0 - duplicates handled

np.int64(0)

## 8. Return before pickup rule validation.


In [11]:
#8. Return before pickup rule validation.
# checking invalid records where Return_TS is before Pickup_TS
(df['Return_TS'] < df['Pickup_TS'])

# counting invalid records
(df['Return_TS'] < df['Pickup_TS']).sum()

# removing invalid records
df = df[df['Return_TS'] >= df['Pickup_TS']] 

# checking after removal
(df['Return_TS'] < df['Pickup_TS']).sum()

np.int64(0)

## 9. Payment method standardization (UPI/CARD/CASH/WALLET).


In [12]:
# 9. Payment method standardization (UPI/CARD/CASH/WALLET).
# df['Payment'].unique()

# define a function to captialize values in payment
def clean_payment(value):
    if isinstance(value,str):
        return value.upper()
    return value

# Apply that function in payment
df['Payment']=df['Payment'].apply(clean_payment)
print(df['Payment'].unique())


# verifying values 
# @pytest.mark.parameterized("val,eval",[
#     ("upi","UPI"),
#     ("Upi","UPI"),
#     ("Cash","CASH"),
#     ("cash","CASH"),
#     ("Wallet","WALLET"),
#     ("wallet","WALLET"),
#     ("card","CARD"),
#     ("Card","CARD")
# ])
# #pytest for that
# def test_clean_payment(value,expval):
#     assert clean_payment(value)==expval


<StringArray>
['WALLET', 'CARD', 'UPI', 'CASH']
Length: 4, dtype: str


In [13]:
df.rename(columns={'Odo_Start': 'Odo_Start_km' ,'Odo_End': 'Odo_End_km'},inplace=True)

## 10. Mileage sanity check (End ≥ Start).


In [14]:
# 10. Mileage sanity check (End ≥ Start).
# how much distance travelled by the vehicle is called mileage
# mileage = odo_start - odo_end

## we have 3 cases were odometer reading can go wrong.
# 1. when mileage < 0 : negitive odometer reading is not possible
# 2. when mileage = 0 : odometer reading is constant
# 3. when mileage > 800: odometer reading shows too high reading in a single day

# df['Mileage']=df['Odo_End']-df['Odo_Start']
# df['Odo_End'].unique()  making Odometer values into int from str

# df['Odo_Start']=df['Odo_Start'].str.replace("km","")
# df['Odo_End']=df['Odo_End'].str.replace("km","")
# df['Odo_Start']=df['Odo_Start'].str.replace(",","")
# df['Odo_End']=df['Odo_End'].str.replace(",","")
# df['Odo_Start']=df['Odo_Start'].astype(int)
# df['Odo_End']=df['Odo_End'].astype(int)

## new column creating , later we can use it for fraud detection
flags = []

for i in range(len(df)):
    start = df.iloc[i]['Odo_Start_km']
    end = df.iloc[i]['Odo_End_km']

    if start is None or end is None:
        flags.append("INVALID")
    elif end < start:
        flags.append("ERROR")
    elif end == start:
        flags.append("ZERO")
    elif (end - start) > 800:
        flags.append("HIGH")
    else:
        flags.append("VALID")

df['mileage_flag'] = flags

df['mileage_flag'].value_counts()

mileage_flag
HIGH     3328
VALID    3144
ERROR     528
Name: count, dtype: int64

## 11. Refueling event detection vs fuel change and distance.


In [15]:

# 11. Refueling event detection vs fuel change and distance.

fuel_flags = []

for i in range(len(df)):
    fuel = df.iloc[i]['Fuel_Fraction']   ## fuel fraction (0-1)
    start = df.iloc[i]['Odo_Start_km']   ## distance calculation
    end = df.iloc[i]['Odo_End_km']
    ## distance moved 0 : but fuel decreasing more - > error
    ## travelled 0 : but at present only LOW fuel
    ## medium
    ## travelled 1: but no fuel (high)
    ## else normal

    if pd.isna(fuel) or pd.isna(start) or pd.isna(end):
        fuel_flags.append("INVALID")
        continue

    fuel_numeric = fuel  # already a float 0-1
    distance = end - start

    if distance < 0:
        fuel_flags.append("ERROR")
    elif distance == 0:
        if fuel_numeric < 0.2:
            fuel_flags.append("LOW_FUEL_NO_USAGE")
        elif fuel_numeric > 0.8:
            fuel_flags.append("POSSIBLE_REFUEL")
        else:
            fuel_flags.append("NORMAL")
    else:
        if fuel_numeric < 0.2:
            fuel_flags.append("HIGH_CONSUMPTION")
        else:
            fuel_flags.append("NORMAL")

df['fuel_flag'] = fuel_flags

## 12. Vehicle availability overlap checks.


In [16]:
# 12. Vehicle availability overlap checks.
overlap_flags = []

for i in range(len(df)):
    
    if i == 0:
        overlap_flags.append("NO_PREVIOUS")
        continue

    curr_vehicle = df.iloc[i]['Vehicle_ID']
    prev_vehicle = df.iloc[i]['Vehicle_ID']

    curr_start = df.iloc[i]['Pickup_TS']
    prev_end = df.iloc[i-1]['Return_TS']

    if curr_vehicle == prev_vehicle:
        if curr_start < prev_end:
            overlap_flags.append("OVERLAP")
        else:
            overlap_flags.append("NO_OVERLAP")
    else:
        overlap_flags.append("NEW_VEHICLE")

df['overlap_flag'] = overlap_flags      ## fleet management


## 13. Damage/incident log linkage to reservation.


In [17]:
#13
# Identify which reservations resulted in vehicle damage or incidents.
damage_cases = df[df["Damage_Flag"] == 1]

print(damage_cases.shape)
damage_cases.head()

# Damage Rate Analysis

damage_rate = df["Damage_Flag"].value_counts(normalize=True)*100
print(damage_rate)

(1245, 33)
Damage_Flag
0    82.214286
1    17.785714
Name: proportion, dtype: float64


## 14. Driver license masking and validation (if present).


In [18]:
# 14
# Handle Missing Values
df["Driver_License"] = df["Driver_License"].fillna("UNKNOWN")

# Handle Invalid Licenses
df.loc[df["Driver_License"] == "INVALID", "Driver_License"] = "UNKNOWN"

#  Validate License Format (DL + 5 digits)
df["License_Valid"] = df["Driver_License"].str.match(r"^DL\d{5}$")

# Mask the License
df["Driver_License"] = df["Driver_License"].apply(
    lambda x: x[:2] + "***" + x[-2:] if x != "UNKNOWN" else "UNKNOWN"
)

## 15. Promo/coupon code validation & expiry checks.


In [19]:
# 15
#  Check Missing promo codes
df["Promo_Code"].isna().sum()

#  Handle Missing Promo Codes
df["Promo_Code"] = df["Promo_Code"].fillna("NO_PROMO")

# Validate Promo Code Format
df["Promo_Valid"] = df["Promo_Code"].str.match(r'^[A-Z]+[0-9]+$')

## 16. Telematics GPS join and jitter smoothing.


In [20]:
# 16
# GPS jitter = small random coordinate fluctuations from telematics devices.

# Jitter smoothing (reduce small GPS noise)
df["GPS_Lat_Smoothed"] = df["GPS_Lat"].round(4)
df["GPS_Lon_Smoothed"] = df["GPS_Lon"].round(4)

print(df[["GPS_Lat","GPS_Lat_Smoothed","GPS_Lon","GPS_Lon_Smoothed"]].head())

        GPS_Lat  GPS_Lat_Smoothed    GPS_Lon  GPS_Lon_Smoothed
1777  15.181000           15.1810  79.808111           79.8081
230   16.814700           16.8147  84.121502           84.1215
5854  23.620000           23.6200  77.476767           77.4768
2939  26.698302           26.6983  84.210011           84.2100
4968  24.578294           24.5783  75.760000           75.7600


## 17. Speeding/harsh events normalization from telematics.


In [21]:
#17
# check columns
df.columns
# check unique values in Harsh_Events
df['Harsh_Events'].unique()
# fill missing values with 0
df['Harsh_Events'] = df['Harsh_Events'].fillna(0)

# normalize → convert to binary (0 and 1)
df['Harsh_Events'] = df['Harsh_Events'].apply(lambda x: 1 if x > 0 else 0)

# final check
df['Harsh_Events'].unique()

array([1, 0])

## 18. PII redaction in notes and feedback.

In [22]:
#18
df.columns
# checking values
df['Notes'].head(10)
df['Customer_Feedback'].head(10)

# checking sample values
df['Notes'].dropna().sample(10)
df['Customer_Feedback'].dropna().sample(10)

# checking presence of phone numbers and emails
df['Notes'].str.contains(r'\d{10}', na=False).sum()
df['Notes'].str.contains(r'\S+@\S+', na=False).sum()

df['Customer_Feedback'].str.contains(r'\d{10}', na=False).sum()
df['Customer_Feedback'].str.contains(r'\S+@\S+', na=False).sum()

#cleaning (masking PII)
import re

def mask_pii(text):
    if pd.isnull(text):
        return text
    
    text = str(text)
    
    # masking phone numbers
    text = re.sub(r'\d{10}', '**********', text)
    
    # mask emails
    text = re.sub(r'\S+@\S+', '*****@*****', text)
    
    return text

# apply cleaning
df['Notes'] = df['Notes'].apply(mask_pii)
df['Customer_Feedback'] = df['Customer_Feedback'].apply(mask_pii)

# final check
df[['Notes', 'Customer_Feedback']].head()

,Notes,Customer_Feedback
1777,scratch on door,Smooth going
230,Contact Rahul **********,good service
5854,Scratch on windows,Smooth going
2939,Contact Rahul **********,email *****@*****
4968,Bad Seats Quality,Good mileage


## 19. Rate plan mapping to master tariffs.


In [23]:
#19
# check values
df['Rate_Plan'].unique()

# clean values (remove spaces + lowercase)
df['Rate_Plan'] = df['Rate_Plan'].astype(str).str.strip().str.lower()

# standardize similar values (if any)
df['Rate_Plan'] = df['Rate_Plan'].replace({
    'std': 'standard',
    'prem': 'premium',
    'eco': 'economy'
})

# Step 4: final check
df['Rate_Plan'].unique()

df['Daily_Rate'].head()
df['Rate_Plan'].unique()

# clean Daily_Rate (if any symbols or text)
df['Daily_Rate'] = pd.to_numeric(df['Daily_Rate'], errors='coerce')

# Step 3: check missing values
df['Daily_Rate'].isnull().sum()

# map Rate_Plan to standard values (if needed)
rate_mapping = {
    'standard': 1500,
    'premium': 2500,
    'economy': 1000
}

# convert Rate_Plan to lowercase
df['Rate_Plan'] = df['Rate_Plan'].astype(str).str.lower()

# fill missing Daily_Rate using Rate_Plan
df['Daily_Rate'] = df['Daily_Rate'].fillna(df['Rate_Plan'].map(rate_mapping))

# final check
df[['Daily_Rate', 'Rate_Plan']].head()

,Daily_Rate,Rate_Plan
1777,1000,economy
230,2500,premium
5854,1000,premium
2939,2000,standard
4968,2000,premium


## 20. Tax/GST computation validation.

In [24]:
#20 
# Step 1: convert columns to numeric
df['Daily_Rate'] = pd.to_numeric(df['Daily_Rate'], errors='coerce')
df['GST_Amount'] = pd.to_numeric(df['GST_Amount'], errors='coerce')
df['Total_Amount'] = pd.to_numeric(df['Total_Amount'], errors='coerce')

# Step 2: fill missing GST values
mean = df['GST_Amount'].mean()
df['GST_Amount'] = df['GST_Amount'].fillna(mean).round(0)

# Step 3: calculate expected total
df['Expected_Total'] = df['Daily_Rate'] + df['GST_Amount']

# Step 4: check mismatch
df['GST_Error'] = df['Total_Amount'] != df['Expected_Total']

# Step 5: view incorrect rows
df[df['GST_Error'] == True]

# Step 6: count errors
df['GST_Error'].sum()

np.int64(1385)

In [25]:
df['Vehicle_ID'].unique()
df['Reservation_ID'].unique()

df['Vehicle_Class'].unique()
df['Booking_TS'].unique()
df['Pickup_TS'].unique()
df['Return_TS'].unique()
df['City'].unique()


<StringArray>
['Hyderabad', 'Mumbai', 'bangalore', 'Delhi', 'Kolkata', 'Chennai']
Length: 6, dtype: str

In [26]:
# df['Harsh_Events'].unique()
# df['fuel_flag'].unique()
# df['Pickup_Lat'].unique()
# df['Customer_Feedback'].unique()  ## customer feedback
# df['Notes'].unique()    ## car feedback
# df['GST_Amount'].unique()
# mean=df['GST_Amount'].mean()
# df['GST_Amount']=df['GST_Amount'].fillna((mean)).round(0)
# df['GST_Amount'].unique()
# df['Total_Amount'].unique()
# df['Rate_Plan'].unique()


In [27]:
# handling null values in longitude, latitude  

# Columns list
cols = ['GPS_Lat','GPS_Lon','Drop_Lat','Drop_Lon','Pickup_Lat','Pickup_Lon']

import numpy as np
import re

def clean_lat_lon_column(df, col):

    # Step 1: Convert to string
    df[col] = df[col].astype(str)

    # Step 2: Remove unwanted characters (keep digits, -, . only)
    df[col] = df[col].apply(lambda x: re.sub(r'[^0-9\.\-]', '', x))

    # Step 3: Convert to numeric
    df[col] = pd.to_numeric(df[col], errors='coerce')

    # Step 4: Create validation flag
    if 'Lat' in col:
        df[col + '_Flag'] = df[col].apply(lambda x: 'INVALID' if (pd.notnull(x) and (x < -90 or x > 90)) else 'VALID')
    else:
        df[col + '_Flag'] = df[col].apply(lambda x: 'INVALID' if (pd.notnull(x) and (x < -180 or x > 180)) else 'VALID')

    # Step 5: Set invalid values to NaN
    if 'Lat' in col:
        df.loc[(df[col] < -90) | (df[col] > 90), col] = np.nan
    else:
        df.loc[(df[col] < -180) | (df[col] > 180), col] = np.nan

    # Step 6: Round values (standard precision)
    df[col] = df[col].round(6)


# Apply to all columns
for col in cols:
    clean_lat_lon_column(df, col)

# Check results
for col in cols:
    print(col, "-> Nulls:", df[col].isnull().sum())

GPS_Lat -> Nulls: 0
GPS_Lon -> Nulls: 0
Drop_Lat -> Nulls: 0
Drop_Lon -> Nulls: 0
Pickup_Lat -> Nulls: 0
Pickup_Lon -> Nulls: 0


In [28]:
df['Damage_Flag'].unique()

array([0, 1])

In [29]:
df['GST_Amount'].isnull().sum() 

np.int64(0)

In [30]:
df.info()

<class 'pandas.DataFrame'>
Index: 7000 entries, 1777 to 584
Data columns (total 45 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Reservation_ID     7000 non-null   str           
 1   Customer_ID_x      7000 non-null   str           
 2   Vehicle_ID         7000 non-null   str           
 3   Vehicle_Class      7000 non-null   str           
 4   Booking_TS         7000 non-null   datetime64[us]
 5   Pickup_TS          7000 non-null   datetime64[us]
 6   Return_TS          7000 non-null   datetime64[us]
 7   City               7000 non-null   str           
 8   Pickup_Lat         7000 non-null   float64       
 9   Pickup_Lon         7000 non-null   float64       
 10  Drop_Lat           7000 non-null   float64       
 11  Drop_Lon           7000 non-null   float64       
 12  Odo_Start_km       7000 non-null   int64         
 13  Odo_End_km         7000 non-null   int64         
 14  Fuel_Fraction      700

In [31]:
df.columns

Index(['Reservation_ID', 'Customer_ID_x', 'Vehicle_ID', 'Vehicle_Class',
       'Booking_TS', 'Pickup_TS', 'Return_TS', 'City', 'Pickup_Lat',
       'Pickup_Lon', 'Drop_Lat', 'Drop_Lon', 'Odo_Start_km', 'Odo_End_km',
       'Fuel_Fraction', 'Damage_Flag', 'GPS_Lat', 'GPS_Lon', 'Max_Speed_kmh',
       'Harsh_Events', 'Payment', 'Daily_Rate', 'Promo_Code', 'GST_Amount',
       'Total_Amount', 'Rate_Plan', 'Customer_ID_y', 'Customer_Feedback',
       'Notes', 'Driver_License', 'mileage_flag', 'fuel_flag', 'overlap_flag',
       'License_Valid', 'Promo_Valid', 'GPS_Lat_Smoothed', 'GPS_Lon_Smoothed',
       'Expected_Total', 'GST_Error', 'GPS_Lat_Flag', 'GPS_Lon_Flag',
       'Drop_Lat_Flag', 'Drop_Lon_Flag', 'Pickup_Lat_Flag', 'Pickup_Lon_Flag'],
      dtype='str')

In [34]:
df.to_csv("cleaned.csv", index=False)